# Phase 3: parse the labels into two tables

Read the XML of every photo in the random sample and build:
- **`images.csv`**: one row per photo (id, country, path, width, height, number of boxes);
- **`boxes.csv`**: one row per box (photo, original code, our class, corners).

Nothing is removed here. Cleaning is Phase 4.

In [1]:
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
from PIL import Image

sys.path.insert(0, str(Path.cwd().parents[1]))
from ml import config

# original code -> our class. D01/D11 are joint cracks (same shape), D43/D44/D50 are not damage we classify.
TO_CLASS = {"D00": "D00", "D01": "D00", "D10": "D10", "D11": "D10", "D20": "D20", "D40": "D40",
            "D43": "none", "D44": "none", "D50": "none"}  # anything else (typos like D0w0) -> "drop"

subset = pd.read_csv(config.PROCESSED / "02_subset" / "subset_images_random.csv")
len(subset)

6000

## 1. Read every XML

In [2]:
image_rows, box_rows = [], []
for r in subset.itertuples():
    folder = config.RAW / r.country / "train"
    root = ET.parse(folder / "annotations" / "xmls" / f"{r.image_id}.xml").getroot()
    boxes = root.findall("object")
    image_rows.append({
        "image_id": r.image_id,
        "country": r.country,
        "path": str((folder / "images" / root.find("filename").text).relative_to(config.RAW)),
        "width": int(root.find("size/width").text),
        "height": int(root.find("size/height").text),
        "n_boxes": len(boxes),
    })
    for o in boxes:
        b, code = o.find("bndbox"), o.find("name").text
        box_rows.append({
            "image_id": r.image_id, "code": code, "cls": TO_CLASS.get(code, "drop"),
            **{k: int(float(b.find(k).text)) for k in ("xmin", "ymin", "xmax", "ymax")},
        })

images, boxes = pd.DataFrame(image_rows), pd.DataFrame(box_rows)
OUT = config.PROCESSED / "03_parse"
OUT.mkdir(parents=True, exist_ok=True)
images.to_csv(OUT / "images.csv", index=False)
boxes.to_csv(OUT / "boxes.csv", index=False)
print(len(images), "photos,", len(boxes), "boxes saved to", OUT)

6000 photos, 8114 boxes saved to /Users/user/Documents/Y4/ML/road-damage-classifier/backend/data/processed/03_parse


In [3]:
images.head()

,image_id,country,path,width,height,n_boxes
0,Czech_000000,Czech,Czech/train/images/Czech_000000.jpg,600,600,0
1,Czech_000001,Czech,Czech/train/images/Czech_000001.jpg,600,600,0
2,Czech_000002,Czech,Czech/train/images/Czech_000002.jpg,600,600,0
3,Czech_000006,Czech,Czech/train/images/Czech_000006.jpg,600,600,1
4,Czech_000009,Czech,Czech/train/images/Czech_000009.jpg,600,600,0


In [4]:
boxes.head()

,image_id,code,cls,xmin,ymin,xmax,ymax
0,Czech_000006,D00,D00,141,405,166,472
1,Czech_000010,D40,D40,213,409,274,441
2,Czech_000010,D10,D10,228,473,327,495
3,Czech_000020,D10,D10,97,477,167,499
4,Czech_000020,D10,D10,110,443,180,465


## 2. Checks

In [5]:
# 1. one row per photo, nothing lost
print("photos:", len(images), "== sample:", len(subset), "->", len(images) == len(subset))
print("duplicate image_ids:", images["image_id"].duplicated().sum())
print("photos with 0 boxes (clean road):", (images["n_boxes"] == 0).sum())
print("boxes per photo, max:", images["n_boxes"].max())

# 2. the XML size matches the real image file
bad = [r.image_id for r in images.itertuples() if Image.open(config.RAW / r.path).size != (r.width, r.height)]
print("XML size differs from the real image:", len(bad))

# 3. boxes make sense
m = boxes.merge(images[["image_id", "width", "height"]], on="image_id")
print("xmax <= xmin or ymax <= ymin:", ((m.xmax <= m.xmin) | (m.ymax <= m.ymin)).sum())
print("outside the photo:", ((m.xmin < 0) | (m.ymin < 0) | (m.xmax > m.width) | (m.ymax > m.height)).sum())

photos: 6000 == sample: 6000 -> True
duplicate image_ids: 0
photos with 0 boxes (clean road): 2369
boxes per photo, max: 13


XML size differs from the real image: 0
xmax <= xmin or ymax <= ymin: 0
outside the photo: 0


## 3. What the codes became

In [6]:
boxes.groupby(["cls", "code"]).size().rename("boxes").to_frame()

boxes
cls  code       
D00  D00    1904
     D01      47
D10  D10    1066
     D11      16
D20  D20    1836
D40  D40    1359
drop D0w0      1
none D43     146
     D44    1051
     D50     688

In [7]:
boxes.groupby(["country" if "country" in boxes else images.set_index("image_id")["country"].reindex(boxes["image_id"]).values, "cls"]).size().unstack(fill_value=0)

cls,D00,D10,D20,D40,drop,none
Czech,724,302,112,150,0,0
India,414,36,551,802,1,299
Japan,813,744,1173,407,0,1586
